# Multi-year NDVI change & forest loss — Cameroon (STAC, auth-free)

The **default** workflow: pull Sentinel-2 L2A baseline and recent composites for a Cameroon forest AOI from the open **Earth Search** STAC catalogue (no Earth Engine account, no sign-in), compute the NDVI change, classify loss/gain, map a before/after split, quantify hectares, and compare the loss total to Global Forest Watch / Hansen tree-cover loss for the same AOI.

**Inspired by** [`giswqs/geemap`](https://github.com/giswqs/geemap). This project makes the runnable, tested path Earth-Engine-free via STAC; the geemap / Earth Engine equivalent is the optional `02_geemap_change.ipynb`. The change-detection core (`eets.indices` / `eets.timeseries` / `eets.change`) is pure numpy and is the same code the unit tests exercise.

## 0. Setup

The STAC path needs the geospatial stack. The pure-numpy core does not.

```bash
conda env create -f environment.yml && conda activate ee-timeseries
# or: pip install -r requirements.txt
```

In [ ]:
# %pip install pystac-client odc-stac rioxarray rasterio matplotlib pyyaml
import matplotlib.pyplot as plt
import yaml

## 1. Configuration

All analysis choices come from `config/aoi.yaml`: the Cameroon bbox, the baseline and recent periods, the index, the change thresholds, and the pixel size.

In [ ]:
with open("../config/aoi.yaml") as fh:
    cfg = yaml.safe_load(fh)

b = cfg["aoi"]["bbox"]
bbox = [b["min_lon"], b["min_lat"], b["max_lon"], b["max_lat"]]
a = cfg["analysis"]
baseline = (cfg["baseline_years"]["start"], cfg["baseline_years"]["end"])
recent = (cfg["recent_years"]["start"], cfg["recent_years"]["end"])
bbox, baseline, recent, a["index"]

## 2. Load baseline & recent composites from STAC

`eets.stac.build_change_inputs` searches Earth Search for Sentinel-2 L2A scenes over the AOI for each period, masks clouds with the scene-classification layer (SCL), computes NDVI per scene, and returns the per-pixel **median** composite for each period — cloud-robust, auth-free.

CLI equivalent: `eets change --config config/aoi.yaml`.

In [ ]:
from eets.stac import build_change_inputs

before, after = build_change_inputs(
    bbox,
    baseline_years=baseline,
    recent_years=recent,
    index=a["index"],
    max_cloud=a["max_cloud"],
    resolution=a["pixel_size_m"],
)
before.shape, after.shape

## 3. Change map, classification, hectares (the pure-numpy core)

This is the tested contribution. `change_map` is `after - before`; `classify_change` splits into loss / stable / gain against the NDVI thresholds; `change_stats` converts pixel counts to hectares.

In [ ]:
from eets.change import change_map, change_stats, classify_change

delta = change_map(before, after)
classified = classify_change(delta, a["loss_thresh"], a["gain_thresh"])
stats = change_stats(classified, a["pixel_size_m"])
stats

## 4. Before / after split map

Baseline NDVI, recent NDVI, and the loss/gain classification side by side.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(before, vmin=0, vmax=1, cmap="YlGn")
ax[0].set_title("Baseline NDVI")
ax[1].imshow(after, vmin=0, vmax=1, cmap="YlGn")
ax[1].set_title("Recent NDVI")
ax[2].imshow(classified, vmin=-1, vmax=1, cmap="RdBu")
ax[2].set_title("Loss (red) / Gain (blue)")
for x in ax:
    x.axis("off")
plt.tight_layout()
plt.show()

## 5. Validate against Global Forest Watch / Hansen

Quantify the forest loss for this AOI from an **external, independent** source and compare it to our Sentinel-2 NDVI-change total. Get the tree-cover-loss area for the same bbox and period from [Global Forest Watch](https://www.globalforestwatch.org/) (the Hansen et al. UMD tree-cover-loss product) — either from the GFW map's analysis tool over your AOI, or the GFW data API — then fill the numbers in below. Agreement within a reasonable margin is the credibility check; disagreement points at threshold sensitivity or cloud contamination.

In [ ]:
# Fill in from GFW / Hansen for this AOI and the recent-vs-baseline window:
gfw_loss_hectares = None  # e.g. tree-cover loss ha, 2020-2024, >30% canopy

s2_loss_hectares = stats["loss_hectares"]
print(f"Sentinel-2 NDVI-change loss : {s2_loss_hectares:.1f} ha")
if gfw_loss_hectares is not None:
    diff = s2_loss_hectares - gfw_loss_hectares
    print(f"GFW / Hansen tree-cover loss: {gfw_loss_hectares:.1f} ha")
    print(
        f"Difference                  : {diff:+.1f} ha "
        f"({100 * diff / gfw_loss_hectares:+.0f}%)"
    )
else:
    print("Set gfw_loss_hectares from Global Forest Watch to validate.")

## 6. Limitations

- **Composite cloud contamination.** Over perennially cloudy equatorial forest, even a median composite can retain haze; widen the period or lower `max_cloud`.
- **Threshold sensitivity.** The loss/gain hectares move with `loss_thresh` / `gain_thresh`; report a sensitivity range, not a single number.
- **NDVI is not tree cover.** A drop can be seasonal phenology or agriculture, not deforestation — hence the GFW / Hansen cross-check.
- **Sentinel-2 starts ~2017.** Longer baselines need Landsat.